# LLM Telephone Game: A Study in Semantic Drift

## 1. Project Overview

This notebook conducts an experiment inspired by the children's game of "Telephone." The goal is to measure **semantic drift**—the degradation and alteration of meaning—when a piece of information is passed sequentially through a chain of different Large Language Models (LLMs).

### The Telephone Game Concept

In the classic game, one person whispers a message to the next, who then whispers it to the next, and so on. The final message is often a comically distorted version of the original. This project applies the same principle to LLMs: one model's answer to a prompt becomes the *only* input for the next model in the chain.

### What is Semantic Drift?

Semantic drift refers to the cumulative change in meaning that occurs with each re-telling. Each LLM, with its unique architecture, training data, and inherent biases, acts as a filter. It re-interprets the input and generates a new output, potentially introducing:
- **Concept Loss:** Key details are omitted.
- **Concept Mutation:** The emphasis or context of information is changed.
- **Hallucinated Additions:** New, unrelated, or incorrect information is invented.

### An Alternative Evaluation Method

Standard LLM benchmarks often test models in isolation. This experiment offers a different perspective by evaluating their **interoperability and fidelity in a dynamic system**. It helps us understand the systemic properties of multi-agent AI systems and the challenges of maintaining information integrity across them.

### Cell 2: Setup and Configuration

This cell handles the foundational setup for our experiment. It performs several key actions:

- **Imports Libraries:** It imports all necessary Python packages, including `os` for environment variables, `pandas` for data handling, `matplotlib` for plotting, and the specific client libraries for the LLMs (`openai`, `anthropic`).
- **Loads Environment Variables:** Using `python-dotenv`, it loads the API keys from your `.env` file into the environment.
- **Initializes Model Clients:** It creates a dictionary called `MODEL_CONFIG` which serves as a central registry for all models. It initializes a client for each provider, cleverly using the `openai.OpenAI` client for all services that offer an OpenAI-compatible API (`google`, `deepseek`, `groq`, and the local `ollama` instance). This streamlines the model calling process significantly.

In [ ]:
import os
import time
import json
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import openai
import anthropic

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

MODEL_CONFIG = {
    "openai": {
        "client": openai.OpenAI(api_key=openai_api_key),
        "model": "gpt-4o"
    },
    "anthropic": {
        "client": anthropic.Anthropic(api_key=anthropic_api_key),
        "model": "claude-3-sonnet-20240229"
    },
    "google": {
        "client": openai.OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/"),
        "model": "gemini-1.5-flash"
    },
    "deepseek": {
        "client": openai.OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1"),
        "model": "deepseek-chat"
    },
    "groq": {
        "client": openai.OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1"),
        "model": "mixtral-8x7b-32768"
    },
    "ollama": {
        "client": openai.OpenAI(base_url='http://localhost:11434/v1', api_key='ollama'),
        "model": "llama3:8b"
    }
}

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Setup Complete: API keys loaded and models configured.")

### Cell 3: Download Local LLM (With Safety Check)

This cell checks if your local Ollama server is running before attempting to pull the model. This prevents the notebook from crashing if you haven't started the Ollama application.

In [ ]:
# Check if Ollama is running
try:
    response = requests.get('http://localhost:11434')
    if response.status_code == 200:
        print("Ollama server is running. Pulling model...")
        !ollama pull llama3:8b
    else:
        print("Warning: Ollama server found but returned unexpected status. Local models may fail.")
except requests.exceptions.ConnectionError:
    print("⚠️ WARNING: Could not connect to Ollama (localhost:11434).")
    print("Please ensure the Ollama app is running before using the 'ollama' provider.")
    # You might want to remove 'ollama' from the chain in Cell 6 if this fails.

### Cell 4: Unified Model Wrapper Function (Updated)

This cell defines the `call_model` function. 

This function  accept a `system_prompt`. This is crucial for the experiment. Without a system prompt telling the model to "paraphrase" or "retell", the models will simply chat back (e.g., "That's a great story!"), which ruins the data on semantic drift.

In [ ]:
def call_model(provider, input_text, temperature=0.7, system_prompt=None):
    """
    Calls the specified LLM provider with the given input text and optional system instruction.

    Args:
        provider (str): The name of the model provider (e.g., 'openai', 'anthropic').
        input_text (str): The text to send to the model.
        temperature (float): The temperature for the model's generation.
        system_prompt (str, optional): Instruction to guide model behavior.

    Returns:
        str: The model's response text.
    """
    if provider not in MODEL_CONFIG:
        raise ValueError(f"Provider '{provider}' not found in MODEL_CONFIG.")

    config = MODEL_CONFIG[provider]
    client = config["client"]
    model_name = config["model"]
    
    print(f"---\nCalling {provider.capitalize()} ({model_name})...")

    try:
        start_time = time.time()
        
        if provider == "anthropic":
            # Anthropic handles system prompts in a specific parameter
            kwargs = {
                "model": model_name,
                "max_tokens": 4096,
                "messages": [{"role": "user", "content": input_text}],
                "temperature": temperature
            }
            if system_prompt:
                kwargs["system"] = system_prompt
                
            response = client.messages.create(**kwargs)
            output_text = response.content[0].text
            
        else:
            # OpenAI-compatible API (OpenAI, Groq, DeepSeek, Google, Ollama)
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": input_text})
            
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=temperature
            )
            output_text = response.choices[0].message.content
        
        end_time = time.time()
        print(f"Response received in {end_time - start_time:.2f} seconds.")
        return output_text.strip()

    except Exception as e:
        print(f"Error calling {provider.capitalize()}: {e}")
        return f"[ERROR: Could not get response from {provider.capitalize()}]"


### Cell 5: Dynamic Seed Prompt Generation

We generate a fresh, complex question to start the chain.

In [ ]:
# Define the request for a new question
question_generation_prompt = ("Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
                            "The question should be suitable for a university-level audience and touch on interdisciplinary concepts. "
                            "Answer only with the question, no explanation.")

print("Generating a new seed prompt from OpenAI...")

# Call OpenAI to generate the prompt
seed_prompt = call_model('openai', question_generation_prompt, temperature=0.8)

print("\n--- Dynamically Generated Seed Prompt ---")
print(seed_prompt)

### Cell 6: Telephone Chain Execution (Corrected Logic)

This cell executes the chain.

Define `TELEPHONE_GAME_INSTRUCTION`. This variable is passed to every model in the chain as a `system_prompt`. It explicitly instructs the model **NOT** to reply conversationally (e.g., "Here is the rewritten text") but to simply output the paraphrased content. This ensures we measure semantic drift, not conversational pleasantries.

In [ ]:
# --- CRITICAL SYSTEM PROMPT ---
# This ensures models behave like telephone players, not helpful assistants.
TELEPHONE_GAME_INSTRUCTION = (
    "You are a player in the game of Telephone. "
    "Your task is to re-state the user's input exactly as you understand it, "
    "using your own words but preserving the core meaning and details. "
    "Do NOT add conversational filler like 'Here is the re-stated text' or 'I understand'. "
    "Just output the content itself."
)

# --- Model Chain Definition ---
model_chain = ["openai", "anthropic", "google", "deepseek", "groq", "ollama"]

# Optional: Define temperature for each model
temperatures = {k: 0.7 for k in model_chain}

# --- Execution Logic ---
telephone_results = []
current_text = seed_prompt

print(f"\nStarting Telephone Chain with order: {model_chain}\n")

for i, provider in enumerate(model_chain):
    input_for_this_step = current_text
    
    # Check if we should skip Ollama if it wasn't running
    if provider == 'ollama':
        try:
            requests.get('http://localhost:11434', timeout=1)
        except:
            print("Skipping Ollama (server not reachable)...")
            continue

    temp = temperatures.get(provider, 0.7)
    
    # CALL THE MODEL WITH THE SYSTEM PROMPT
    output_text = call_model(
        provider, 
        input_for_this_step, 
        temperature=temp, 
        system_prompt=TELEPHONE_GAME_INSTRUCTION
    )

    telephone_results.append({
        "step": i + 1,
        "provider": provider,
        "input_text": input_for_this_step,
        "output_text": output_text
    })
    
    # The output of this step becomes the input for the next
    current_text = output_text

# --- Display Results ---
results_df = pd.DataFrame(telephone_results)
print("\nTelephone Chain Complete.")

# Display the final output for quick review
final_output = results_df.iloc[-1]['output_text']
print("\n--- Final Output ---")
print(final_output)

### Cell 7: Reconstruction Phase

We ask GPT-4o to act as a detective and guess the original question based ONLY on the final, distorted output.

In [ ]:
reconstruction_prompt_template = (
    "The following text is the final output from a long chain of AI models passing a message to each other. "
    "The original message has been lost. Your task is to analyze this final text and reconstruct the most likely *original question or instruction* that started the chain. "
    "Be concise and focus on the core concepts present in the text."
    "\n\n--- FINAL TEXT ---\n{final_output}"
    "\n\n--- RECONSTRUCTED ORIGINAL QUESTION ---"
)

final_output_text = telephone_results[-1]['output_text']
reconstruction_prompt = reconstruction_prompt_template.format(final_output=final_output_text)

# We use a capable model for the reconstruction task
reconstructed_prompt = call_model('openai', reconstruction_prompt, temperature=0.5)

print("--- Original Seed Prompt ---")
print(seed_prompt)
print("\n--- Reconstructed Prompt ---")
print(reconstructed_prompt)

### Cell 8: Semantic Drift Evaluation

We perform both embedding-based similarity checks and LLM-as-a-judge qualitative checks.

In [ ]:
# --- 1. Embedding-based Similarity Calculation ---
print("Calculating semantic similarity using embeddings...")

# Collate all texts: original, each step's output, and the final reconstruction
all_texts = [seed_prompt] + results_df['output_text'].tolist() + [reconstructed_prompt]
all_labels = ['Original Seed'] + [f'Step {i+1} ({p})' for i, p in enumerate(results_df['provider'])] + ['Reconstructed']

# Generate embeddings
embeddings = embedding_model.encode(all_texts)

# Calculate cosine similarity with the original seed prompt's embedding
original_embedding = embeddings[0].reshape(1, -1)
similarities = cosine_similarity(original_embedding, embeddings)[0]

# Add similarity scores to the results dataframe
# We shift the scores to align with the results_df index
similarity_scores = similarities[1:len(results_df)+1] 
results_df['similarity_to_original'] = similarity_scores

# --- 2. LLM-based Qualitative Evaluation ---
print("\nPerforming LLM-based qualitative evaluation...")

evaluator_prompt_template = """
You are an expert AI analyst. Your task is to evaluate the semantic drift between an original text and a subsequent version that has been processed by other AI models.

Please compare the 'Original Prompt' with the 'Model Output' and provide a structured analysis in JSON format.

**Original Prompt:**
{original_text}

**Model Output:**
{model_output}

**Evaluation Criteria:**
1.  **Concept Loss (0-10):** How much of the core information from the original is missing? (0 = no loss, 10 = total loss).
2.  **Concept Mutation (0-10):** How much has the meaning or emphasis of the concepts been altered? (0 = no mutation, 10 = completely different meaning).
3.  **Hallucinated Additions (0-10):** How much new, irrelevant, or incorrect information has been added? (0 = none, 10 = mostly hallucinations).
4.  **Core Idea Survival Score (0-100):** A holistic score representing the percentage of the original core idea that remains intact.

**Reasoning:**
Provide a brief, one-sentence explanation for your scores.

**Output Format (JSON only, no markdown formatting):
{{
  "concept_loss": <score>,
  "concept_mutation": <score>,
  "hallucinated_additions": <score>,
  "core_idea_survival_score": <score>,
  "reasoning": "<your brief reasoning>"
}}
"""

llm_evaluations = []
for index, row in results_df.iterrows():
    print(f"Evaluating output from Step {row['step']} ({row['provider']})...")
    eval_prompt = evaluator_prompt_template.format(original_text=seed_prompt, model_output=row['output_text'])
    
    # Use a capable model for evaluation
    evaluation_response = call_model('openai', eval_prompt, temperature=0.2)
    
    try:
        # Clean the response to ensure it's valid JSON
        cleaned_response = evaluation_response.strip().replace('`', '').replace('json', '')
        eval_json = json.loads(cleaned_response)
        llm_evaluations.append(eval_json)
    except json.JSONDecodeError:
        print(f"  - Failed to parse JSON from evaluator for step {row['step']}. Appending empty dict.")
        llm_evaluations.append({})

# Merge LLM evaluations into the main dataframe
eval_df = pd.DataFrame(llm_evaluations)
results_df = pd.concat([results_df, eval_df], axis=1)

print("\nEvaluation Complete.")

# Display the full results table
from IPython.display import display
print("\n--- Combined Evaluation Results ---")
# To prevent wide columns from truncating
pd.set_option('display.max_colwidth', 100)
display(results_df[['step', 'provider', 'similarity_to_original', 'core_idea_survival_score', 'concept_loss', 'concept_mutation', 'hallucinated_additions', 'reasoning']])

### Cell 9: Visualization of Semantic Drift

We visualize the degradation of meaning over time.

In [ ]:
# Ensure the dataframe has the required columns before plotting
if 'similarity_to_original' not in results_df.columns or 'core_idea_survival_score' not in results_df.columns:
    print("Evaluation columns not found. Please run the evaluation cell (7) first.")
else:
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax1 = plt.subplots(figsize=(14, 7))

    # X-axis labels
    step_labels = [f"Step {r['step']}\n({r['provider']})" for i, r in results_df.iterrows()]
    steps = results_df['step']

    # Plot 1: Semantic Similarity (Embedding-based)
    color = 'tab:blue'
    ax1.set_xlabel('Model Chain Step')
    ax1.set_ylabel('Semantic Similarity (Cosine)', color=color)
    ax1.plot(steps, results_df['similarity_to_original'], 'o-', color=color, label='Embedding Similarity')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_xticks(steps)
    ax1.set_xticklabels(step_labels, rotation=45, ha="right")

    # Plot 2: Core Idea Survival (LLM-based)
    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
    color = 'tab:green'
    ax2.set_ylabel('Core Idea Survival Score (%)', color=color)
    ax2.plot(steps, results_df['core_idea_survival_score'], 's--', color=color, label='LLM-based Survival Score')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.set_ylim(0, 105) # Survival score is 0-100

    # Final Touches
    fig.tight_layout() # adjust plot to ensure everything fits without overlapping
    plt.title('Semantic Drift Across LLM Chain', fontsize=16)
    fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
    plt.grid(True)
    plt.show()

    # --- Secondary Plot: Breakdown of LLM Evaluation ---
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Plotting the three components of drift
    ax.plot(steps, results_df['concept_loss'], 'o-', label='Concept Loss', color='tab:red')
    ax.plot(steps, results_df['concept_mutation'], 's-', label='Concept Mutation', color='tab:orange')
    ax.plot(steps, results_df['hallucinated_additions'], '^-', label='Hallucinated Additions', color='tab:purple')

    ax.set_xlabel('Model Chain Step')
    ax.set_ylabel('Evaluation Score (0-10)')
    ax.set_xticks(steps)
    ax.set_xticklabels(step_labels, rotation=45, ha="right")
    ax.set_ylim(0, 11)
    ax.set_title('LLM-based Evaluation: Drift Components')
    ax.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### Cell 10: Results Summary and Persistence

We calculate the final metrics and save the entire experiment to a CSV file so you don't lose the data.

In [ ]:
# --- Final Summary Table ---
print("--- Final Summary Table ---")
display(results_df[['step', 'provider', 'similarity_to_original', 'core_idea_survival_score']])

# --- Final Verdict ---
initial_similarity = 1.0
final_similarity = results_df.iloc[-1]['similarity_to_original']
degradation = (initial_similarity - final_similarity) * 100

initial_survival = 100
final_survival = results_df.iloc[-1]['core_idea_survival_score']
survival_loss = initial_survival - final_survival

reconstruction_similarity = cosine_similarity(embedding_model.encode([seed_prompt]), embedding_model.encode([reconstructed_prompt]))[0][0]

print("\n--- Final Verdict ---")
print(f"Overall Semantic Degradation (based on embeddings): {degradation:.2f}% loss of similarity.")
print(f"Overall Core Idea Loss (based on LLM eval): {survival_loss:.2f}% loss of core idea.")
print(f"Reconstruction Quality: The reconstructed prompt had a {reconstruction_similarity:.2f} similarity score with the original seed.")

if final_similarity > 0.8 and final_survival > 75:
    verdict = "The core meaning survived the chain remarkably well."
elif final_similarity > 0.6 and final_survival > 50:
    verdict = "Significant semantic drift was observed, but the core idea is still recognizable."
else:
    verdict = "The message was heavily distorted, with substantial loss of the original meaning."

print(f"\nConclusion: {verdict}")

# --- Save Results ---
timestamp = time.strftime("%Y%m%d-%H%M%S")
filename = f'telephone_game_results_{timestamp}.csv'
results_df.to_csv(filename, index=False)
print(f"\n✅ Experiment complete. Results saved to '{filename}'.")